<h3 style="color:#6FA8DC; font-weight:bold">05 — Advanced Outlier Detection: Isolation Forest</h3>

This notebook introduces a machine-learning approach to outlier detection.

We will learn:
- Why univariate methods are not enough
- Multivariate outliers
- Isolation Forest intuition
- `contamination`
- `n_estimators`
- `random_state`
- `fit_predict()`
- `decision_function()`
- Visualizing detected outliers
- Multiple-feature example
- When Isolation Forest is useful
- Important production considerations

<h5 style="color:#78B89A; font-weight:bold;">Why do we need multivariate outlier detection?</h5>

Univariate methods inspect one feature at a time.

But sometimes an observation is normal in every individual column and unusual only because of the **combination of features**.

Example:

```text
Age     Income
25      50,000
26      55,000
27      60,000
25      5,000,000  ← unusual combination
```

A multivariate method can detect relationships like this.

<h5 style="color:#78B89A; font-weight:bold;">Isolation Forest → intuition</h5>

Isolation Forest is based on the idea that anomalies are often **easier to isolate** than normal observations.

Imagine repeatedly splitting the data:

```text
Normal observations
→ need many splits to isolate

Anomalous observations
→ often become isolated quickly
```

The algorithm builds many random trees and combines the results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest

np.random.seed(42)

normal_data = np.random.normal(
    loc=[50, 50],
    scale=[8, 8],
    size=(300, 2)
)

outlier_data = np.array([
    [90, 10],
    [10, 90],
    [100, 100],
    [5, 5],
    [110, 20]
])

X = np.vstack([normal_data, outlier_data])

df = pd.DataFrame(X, columns=["feature_1", "feature_2"])

df.head()

<h5 style="color:#78B89A; font-weight:bold;">Visualize the data first</h5>

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["feature_1"], df["feature_2"])
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Multivariate Dataset")
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">Train Isolation Forest</h5>

`contamination` represents the expected proportion of outliers in the data.

For learning, we will use `0.02` as an example.

In a real project, this value should be selected using domain knowledge, validation, or a suitable detection strategy.

In [ ]:
model = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42
)

model.fit(df[["feature_1", "feature_2"]])

<h5 style="color:#78B89A; font-weight:bold;">Predict outliers</h5>

Isolation Forest returns:

```text
 1  → inlier / normal observation
-1  → outlier
```

In [ ]:
df["outlier_label"] = model.predict(
    df[["feature_1", "feature_2"]]
)

df["is_outlier"] = df["outlier_label"] == -1

print(df["outlier_label"].value_counts())
display(df[df["is_outlier"]].head(20))

<h5 style="color:#78B89A; font-weight:bold;">Visualize detected outliers</h5>

In [ ]:
plt.figure(figsize=(8, 5))

normal = df[~df["is_outlier"]]
outliers = df[df["is_outlier"]]

plt.scatter(normal["feature_1"], normal["feature_2"], label="Normal")
plt.scatter(outliers["feature_1"], outliers["feature_2"], label="Outlier", marker="x", s=100)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Isolation Forest Detection")
plt.legend()
plt.show()

<h5 style="color:#78B89A; font-weight:bold;">decision_function() → anomaly score</h5>

`decision_function()` gives a score that can be used to understand how strongly observations are considered anomalous.

More negative values indicate stronger anomaly behavior for Isolation Forest.

In [ ]:
df["anomaly_score"] = model.decision_function(
    df[["feature_1", "feature_2"]]
)

df.sort_values("anomaly_score").head(10)

<h5 style="color:#78B89A; font-weight:bold;">Remove detected outliers → only after investigation</h5>

In [ ]:
X_clean = df[df["outlier_label"] == 1].drop(
    columns=["outlier_label", "is_outlier", "anomaly_score"]
)

print("Original rows:", len(df))
print("Rows after filtering:", len(X_clean))

<h5 style="color:#78B89A; font-weight:bold;">Isolation Forest with a real ML dataset → breast cancer features</h5>

Here we use a multi-feature dataset from scikit-learn to demonstrate that Isolation Forest can work with many numerical columns.

This is a demonstration of anomaly detection, not a claim that the original dataset's unusual observations are errors.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

X_real = data.data

print("Shape:", X_real.shape)
display(X_real.head())

In [ ]:
iso_real = IsolationForest(
    n_estimators=200,
    contamination=0.03,
    random_state=42
)

labels = iso_real.fit_predict(X_real)

print(pd.Series(labels).value_counts())

<h5 style="color:#78B89A; font-weight:bold;">Important: Isolation Forest is not the same as IQR</h5>

```text
IQR
→ usually univariate
→ statistical rule
→ easy to explain

Isolation Forest
→ multivariate
→ machine-learning based
→ useful for complex feature relationships
```

<h5 style="color:#78B89A; font-weight:bold;">When should Isolation Forest be considered?</h5>

Useful when:
- several features interact
- simple univariate rules miss unusual combinations
- the dataset has many numerical features
- you need an ML-based anomaly detector

Be careful when:
- the dataset is very small
- the feature space has unusual structure
- domain knowledge gives better rules
- the detected observations are genuine rare cases

<h5 style="color:#78B89A; font-weight:bold;">Production considerations ⭐</h5>

Do not automatically delete every observation predicted as an anomaly.

A safer production workflow is:

```text
Training data
      ↓
EDA + domain knowledge
      ↓
Fit anomaly detector
      ↓
Validate detection behavior
      ↓
Save fitted detector
      ↓
New data
      ↓
Same detector
      ↓
Flag / investigate / treat
```

Also remember that anomaly detection can be a **business decision**, not just a preprocessing operation.

<h3 style="color:#6FA8DC; font-weight:bold">Outlier Detection Series — Where We Are</h3>

```text
01 → Outliers Introduction
02 → Z-Score
03 → IQR
04 → Percentile / Quantile
05 → Isolation Forest / Multivariate Detection
```

### Quick selection

```text
One numerical feature
       ↓
   ┌───┴────┐
   ↓        ↓
 Normal   Skewed
   ↓        ↓
Z-Score    IQR
       or Percentile

Multiple features
       ↓
Multivariate methods
       ↓
Isolation Forest / LOF / One-Class SVM
```

⭐ Always distinguish:

**unusual ≠ wrong**

An outlier detector identifies observations for investigation; it does not automatically decide that the observation should be deleted.